# 🎵 ARIS: 从零到一完整教程 (From Scratch to Acoustic Manipulation)
### 神经网络源–滤波器可微声码器：语音学实验刺激生成全流程

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/N1r/ARIS_nsf/blob/main/notebooks/ARIS_Tutorial_and_Workflow.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-ARIS__nsf-blue.svg)](https://github.com/N1r/ARIS_nsf)
[![Interactive Demo](https://img.shields.io/badge/Web-Listening_Demo-green.svg)](https://n1r.github.io/ARIS_nsf/)

---

> ### 💡 计算机零基础？点此极简上手！
> 如果您是语音学、语言学、心理学或认知科学背景，对代码或终端命令不熟悉，完全不用担心：
> 1. **全自动运行**：点击 Google Colab 顶部菜单栏的 **“代码执行程序” (Runtime) ➔ “全部运行” (Run all)**。
> 2. **按顺序浏览**：代码会自动完成环境配置、下载音频、数据准备、演示训练，并生成试听音频与语谱图。
> 3. **随时试听**：向下滚动即可直接点击播放按钮试听原声、重建声音与不同参数操控后的变体声音！
> 4. **网页图形工作台**：第 7 节会生成一个类似 Praat 的交互式网页链接，可以直接在浏览器里用鼠标拖动滑块调音。

---

### 📋 本教程完整工作流概览：
$$\text{1. 环境安装} \longrightarrow \text{2. 下载音频} \longrightarrow \text{3. 数据准备 (切分/F0)} \longrightarrow \text{4. 模型训练} \longrightarrow \text{5. 成果试听与语谱图} \longrightarrow \text{6. 语音学操控 (Manipulation)}$$


## 步骤 1. 🛠️ 环境安装 (Environment Setup)

Google Colab 提供了免费的 GPU 加速（T4/L4）。我们首先检查显卡，并使用单文件包管理器 `uv` 在十几秒内极速安装 ARIS 及其全部依赖库。


In [ ]:
# 1.1 检查云端 GPU 状态
!nvidia-smi


In [ ]:
# 1.2 克隆仓库并使用 uv 极速安装
import os

if not os.path.exists("pyproject.toml"):
    !git clone https://github.com/N1r/ARIS_nsf.git
    %cd ARIS_nsf

# 安装 uv 并通过 pyproject.toml 同步全部依赖（只需约15秒）：
!pip install -q uv
!uv pip install --system -e ".[all]"
print("✓ ARIS 及全部音频/训练/界面依赖安装就绪！")


In [ ]:
# 1.3 运行内置诊断工具检查环境
import aris

print(f"ARIS 版本: {aris.__version__}\n")
for item in aris.doctor():
    mark = "✓ [OK]" if item.ok else "✗ [--]"
    print(f"{mark} {item.name:12} : {item.detail} ({item.required_for})")


---
## 步骤 2. 📥 下载音频数据 (Download Audio Data)

为了让您可以立即上手，我们从 GitHub Release 下载官方提供的配套语音包（包含普通话女声 F024 的 16 kHz 语音录音与训练就绪的模型，约 65 MB）：


In [ ]:
# 2.1 下载并解压音频数据包 (ZIP)
import os

if not os.path.exists("demo_f024"):
    !curl -LO https://github.com/N1r/ARIS_nsf/releases/download/v0.1.0/aris_f024_demo.zip
    !unzip -q aris_f024_demo.zip
    print("✓ 音频数据与模型已解压至 demo_f024/")
else:
    print("✓ demo_f024/ 数据目录已存在。")


In [ ]:
# 2.2 试听原始音频录音
import glob

from IPython.display import Audio, display

raw_audios = sorted(glob.glob("demo_f024/dataset/audio/*.wav"))
print(f"共发现 {len(raw_audios)} 条训练音频，试听第 1 条原始发音：")
print("文件名:", raw_audios[0])
display(Audio(raw_audios[0]))


---
## 步骤 3. ✂️ 数据准备 (Data Preparation)

在实际科研中，从一段长录音到可以训练的语音学数据集，通常包含三个关键操作：
1. **静音分句 (`aris split`)**：将连续说话切分成 2–6 秒的单句短句。
2. **特征准备 (`aris prepare`)**：将音频统一重采样到目标采样率（16 kHz），使用 WORLD 算法提取精确的基频 $F_0$ 轮廓，并按比例划分为训练集、验证集和测试集。
3. **数据校验 (`aris validate`)**：检查音频削波、静音段和文件完整性。


In [ ]:
# 3.1 演示数据准备流程（提取 F0 与索引构建）
# 我们基于提取出的音频演示 prepare 流程：
!aris prepare demo_f024/dataset/audio data/my_prepared_data --sample-rate 16000 --f0-method pyworld
print("✓ 数据准备完成！F0 特征已提取，生成了 data/my_prepared_data/manifest.csv")


In [ ]:
# 3.2 校验数据集完整性
!aris validate data/my_prepared_data
print("✓ 数据校验通过，可以直接用于训练！")


---
## 步骤 4. 🚀 模型配置与训练 (Model Training)

ARIS 采用先进的 `aria-golf` 架构（基于可微时变线性预测与声门气流波形）：
- **声源分支（Source）**：精确模拟声带振动波形与声门倾斜度（$R_d$）。
- **声道滤波器（Filter）**：基于全极点共振峰滤波器，可独立控制各共振峰（$F_1, F_2$）。

我们在 GPU 上快速演示 50 步的训练循环（仅需不到 1 分钟）：


In [ ]:
# 4.1 初始化实验配置
!aris init-experiment data/my_prepared_data experiments/my_quick_experiment --model aria-golf --batch-size 16 --max-steps 50
print("✓ 实验初始化完成：配置文件已写入 experiments/my_quick_experiment/")


In [ ]:
# 4.2 启动快速训练循环演示（展示 GPU 训练过程）
!aris train experiments/my_quick_experiment --trainer.max_steps=50
print("✓ 训练演示完成！在真实研究中，通常训练 10,000~30,000 步（约 1~2 小时）即可达到完美收敛。")


---
## 步骤 5. 📊 可视化成果与试听评估 (Visualize Results & Evaluation)

现在，我们使用训练好的收敛模型对保留的测试音频进行**分析重建**，并对比原声与合成声的波形与语谱图：


In [ ]:
# 5.1 使用 checkpoint 进行语音重建推理
!aris synthesize demo_f024/experiment demo_f024/experiment/runs/checkpoints/last.ckpt out/demo_recon
print("✓ 重建完成！合成音频保存在 out/demo_recon/")


In [ ]:
# 5.2 原声 vs 重建语音 并排试听 (A/B 对比)
target_stem = "F024_bian4"
recon_wav = glob.glob(f"out/demo_recon/{target_stem}*.wav")[0]
# 找到对应的原声
orig_wav = glob.glob(f"demo_f024/dataset/audio/{target_stem}*.wav")[0]

print("🔊 【原声】录音室原始发音：")
display(Audio(orig_wav))

print("🔊 【ARIS 重建】神经网络声码器重建发音：")
display(Audio(recon_wav))


In [ ]:
# 5.3 绘制时频对齐语谱图对比
import matplotlib.pyplot as plt
import soundfile as sf


def plot_comparison(wav_a, wav_b, title_a="原始录音", title_b="ARIS 重建"):
    y_a, sr_a = sf.read(wav_a)
    y_b, sr_b = sf.read(wav_b)
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    
    axes[0].specgram(y_a, Fs=sr_a, NFFT=512, noverlap=384, cmap="viridis")
    axes[0].set_title(title_a, fontsize=12, fontweight="bold")
    axes[0].set_ylabel("频率 (Hz)")
    axes[0].set_ylim(0, 5000)
    
    axes[1].specgram(y_b, Fs=sr_b, NFFT=512, noverlap=384, cmap="viridis")
    axes[1].set_title(title_b, fontsize=12, fontweight="bold")
    axes[1].set_ylabel("频率 (Hz)")
    axes[1].set_xlabel("时间 (s)")
    axes[1].set_ylim(0, 5000)
    
    plt.tight_layout()
    plt.show()

plot_comparison(orig_wav, recon_wav, "【原始录音】语谱图", "【ARIS 可微重建】语谱图")


---
## 步骤 6. 🎛️ 语音学生学参数操控 (Manipulation)

这是 ARIS 最核心的能力：**在保证所有其他语音声学维度完全固定的前提下，单独且精确地改变特定声学特征**。

常见语音学操控维度：
- `f1_scale=1.2`：第一共振峰 $F_1$ 升高 20%（模拟舌位降低、开口度增大，例如元音向半开/开元音方向偏移）。
- `pitch_semitones=-4`：基频 $F_0$ 整体降低 4 个半音（模拟基频/声调下移）。
- `glottal_rd_scale=1.6`：声门波形参数增加（模拟气声/耳语感 breathy voice）。
- `glottal_rd_scale=0.6`：声门波形参数降低（模拟声带紧闭/嘎裂声 creaky voice）。


In [ ]:
# 6.1 批量生成成对实验刺激
!aris manipulate demo_f024/experiment demo_f024/experiment/runs/checkpoints/last.ckpt out/demo_stimuli \
  --variant "f1_up:f1_scale=1.2" \
  --variant "pitch_down:pitch_semitones=-4" \
  --variant "breathy:glottal_rd_scale=1.6" \
  --variant "creaky:glottal_rd_scale=0.6"

print("✓ 全部实验刺激已生成在 out/demo_stimuli/ 目录下！")


In [ ]:
# 6.2 试听操控刺激对比
f1_wav = glob.glob(f"out/demo_stimuli/f1_up/{target_stem}*.wav")[0]
pitch_wav = glob.glob(f"out/demo_stimuli/pitch_down/{target_stem}*.wav")[0]
breathy_wav = glob.glob(f"out/demo_stimuli/breathy/{target_stem}*.wav")[0]
creaky_wav = glob.glob(f"out/demo_stimuli/creaky/{target_stem}*.wav")[0]

print("🔊 1. 基准重建声音 (Baseline):")
display(Audio(recon_wav))

print("🔊 2. 共振峰 F1 抬高 (+20% F1 Scale，开口度变大):")
display(Audio(f1_wav))

print("🔊 3. 基频降低 (-4 个半音):")
display(Audio(pitch_wav))

print("🔊 4. 气声音质 (Breathy Voice, Rd = 1.6):")
display(Audio(breathy_wav))

print("🔊 5. 紧喉/嘎裂音质 (Creaky Voice, Rd = 0.6):")
display(Audio(creaky_wav))


In [ ]:
# 6.3 语谱图直观查看 F1 共振峰抬高与音高变化
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, wav, title in zip(
    axes,
    [recon_wav, f1_wav, pitch_wav],
    ["基准重建 (Baseline)", "第一共振峰升高 (+20% F1 Scale)", "基频整体降低 (-4 Semitones)"]
):
    y, sr = sf.read(wav)
    ax.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap="magma")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("频率 (Hz)")
    ax.set_ylim(0, 4500)

axes[-1].set_xlabel("时间 (s)")
plt.tight_layout()
plt.show()


---
## 步骤 7. 🌐 网页交互式工作台 (Interactive Web Studio)

不想敲命令的话，可以在云端直接启动内置的 Gradio 网页工作室。它会自动生成一个公网访问链接（`https://xxxx.gradio.live`），点击打开即可在浏览器里：
- 用鼠标拖动所有声学参数滑块
- 一键生成等距连续统（Continuum Builder）
- 原声与变体 A/B 盲听与语谱图对比


In [ ]:
# 启动可视化 Studio（点击输出的 public URL 即可在本地浏览器操作）：
# 运行时会持续保持服务，按单元格左侧的停止按钮即可退出。

import aris

aris.launch_studio(workspace=".", share=True, open_browser=False)


---
## 步骤 8. 📦 打包与下载实验刺激 (Export Stimuli)

ARIS 自动为每一次操控生成严格的学术元数据记录文件（`manipulation.json`），记录了模型哈希、数据指纹与具体参数，确保 100% 的计算可复现性。一键打包即可下载供心理学实验（如 E-Prime, PsychoPy, jsPsych）直接使用：


In [ ]:
# 打包生成的实验刺激与元数据
!zip -q -r aris_experiment_stimuli.zip out/demo_stimuli out/demo_recon
print("✓ 刺激已压缩打包为 aris_experiment_stimuli.zip")

# 如果在 Google Colab 中运行，取消下面两行的注释可自动弹出浏览器下载：
# from google.colab import files
# files.download("aris_experiment_stimuli.zip")
